This script takes the per-run MEG manifest (MEG_manifest.csv), resolves each run’s .fif path under MEG_DATA_DIR, then for each subject:
- Ensures the subject’s FreeSurfer recon directory exists under SUBJECTS_DIR.
- Builds watershed BEM surfaces and writes a BEM solution file to the coregistration output folder.
- For each MEG run/session: loads the raw FIF, performs MNE Coregistration (fiducial fit + ICP), writes a session-specific '*_trans.fif' (transformation affine) file, and stores the transform path back into 'MEG_runs' as 'trans_fullpath'.
- Writes a 'BEM_issues.txt' summary list of subjects that failed any BEM/coreg steps.

[Runtime: approx. 1-2 min per MEG scan file]

-------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, datetime, subprocess, shutil, re
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import mne
from mne.bem import make_watershed_bem
import warnings

In [ ]:
##### SET UP ENVIRONMENTAL VARIABLES FOR FREESURFER:
freesurfer_config = config.get("freesurfer", {})
FREESURFER_HOME = Path(freesurfer_config.get("home", "/opt/freesurfer-7.4.1")).expanduser()
FS_LICENSE = Path(freesurfer_config.get("license", FREESURFER_HOME / "license.txt")).expanduser()
SUBJECTS_DIR = Path(freesurfer_config.get("subjects_dir", FREESURFER_HOME / "subjects")).expanduser()
CHECK_FS_VERSION = bool(freesurfer_config.get("check_version", True))
if not FREESURFER_HOME.exists():
    raise FileNotFoundError(f"FREESURFER_HOME not found: {FREESURFER_HOME}")
if not FS_LICENSE.exists():
    raise FileNotFoundError(f"FreeSurfer license file not found: {FS_LICENSE}")
if not SUBJECTS_DIR.exists():
    raise FileNotFoundError(f"FreeSurfer SUBJECTS_DIR not found: {SUBJECTS_DIR}")
os.environ["FREESURFER_HOME"] = str(FREESURFER_HOME)
os.environ["FS_LICENSE"] = str(FS_LICENSE)
os.environ["SUBJECTS_DIR"] = str(SUBJECTS_DIR)
fs_bin_dir = FREESURFER_HOME / "bin"
os.environ["PATH"] = f"{fs_bin_dir}:{os.environ.get('PATH', '')}"

subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# --------------------------------------------------------------------
### SET PARAMETERS:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

OVERWRITE_COREG = config['overwrite_coregistration']
OVERWRITE_WATERSHED = config['overwrite_watershed']

BEM_CONNECTIVITY = config['BEM_parameters']['BEM_connectivity']
BEM_ICO = config['BEM_parameters']['BEM_ICO']
BEM_FLOODING = config['BEM_parameters']['BEM_flooding']

COREG_ITER_NUM = config['coregistration_parameters']['num_coregistration_iterations']


# --------------------------------------------------------------------
### SET PATHS:

ROOT_DIR = Path(config['root_output_directory'])

### INPUTS:
RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
MEG_DATA_DIR = config['MEG_data_directory']
MEG_PARAMETERS_PATH = Path(ROOT_DIR) / 'MEG_manifest.csv'

### OUTPUTS:

COREG_OUTPUT_DIR = ROOT_DIR / config['coreg_output_dir']
COREG_OUTPUT_DIR.mkdir(exist_ok=True)


# --------------------------------------------------------------------
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
MEG_runs = pd.read_csv(MEG_PARAMETERS_PATH)

In [ ]:
### DIAGNOSTIC SUBSETTING (if enabled):
if type(SUBSET) == int and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    MEG_runs = MEG_runs.head(SUBSET).copy()
    display(MEG_runs)

---------

First, retrieve full filepaths for all target MEG files:

In [ ]:
# --------------------------------------------------------------------
### RESOLVE MEG FILE PATHS (recursive search under MEG_DATA_DIR)
# --------------------------------------------------------------------

# Initialize column
MEG_runs['MEG_fullpath'] = pd.NA

all_meg_fif_paths = []

for root, dirnames, filenames in os.walk(MEG_DATA_DIR):
    for filename in filenames:
        if filename.endswith('.fif'):
            all_meg_fif_paths.append(Path(root) / filename)

print(f"[scan] Found {len(all_meg_fif_paths)} .fif files under {MEG_DATA_DIR}")

# Build an index from base name (without '.fif') to path
MEG_file_index = {}
duplicate_basenames = []

for path in all_meg_fif_paths:
    base_name = path.name[:-4]  # strip ".fif"
    if base_name in MEG_file_index and MEG_file_index[base_name] != path:
        duplicate_basenames.append(base_name)
    else:
        MEG_file_index[base_name] = path

if duplicate_basenames:
    print("[warn] Multiple .fif files found for some base names:")
    for base_name in sorted(set(duplicate_basenames)):
        print(f"   - {base_name}")
    if HARD_STOP:
        raise RuntimeError(
            "Duplicate MEG base names found under MEG_DATA_DIR; "
            "please resolve these conflicts before continuing.")

# Map each MEG_runs row to a concrete path
missing_meg_entries = []

for dataframe_index, run_row in MEG_runs.iterrows():
    meg_filename_base = run_row['MEG_filename']  # BIDS-style base, no extension
    if meg_filename_base in MEG_file_index:
        MEG_runs.at[dataframe_index, 'MEG_fullpath'] = str(MEG_file_index[meg_filename_base])
    else:
        missing_meg_entries.append((dataframe_index, run_row['subject_ID'], meg_filename_base))

if missing_meg_entries:
    print("[warn] Missing MEG files for the following runs (row_index, subject_ID, MEG_filename):")
    for dataframe_index, subject_ID, meg_filename_base in missing_meg_entries[:10]:
        print(f"   - {dataframe_index}: {subject_ID} | {meg_filename_base}")
    print(f"[warn] Total missing MEG runs: {len(missing_meg_entries)}")

    if HARD_STOP:
        raise FileNotFoundError(
            f"Missing MEG .fif files for {len(missing_meg_entries)} run(s); "
            "see warnings above.")
    else:
        # Drop missing rows from MEG_runs when HARD_STOP is False
        missing_indices = [idx for idx, _, _ in missing_meg_entries]
        MEG_runs = MEG_runs.drop(index=missing_indices).reset_index(drop=True)
        print(f"[info] Dropped {len(missing_indices)} run(s) with missing MEG files; "
              f"{len(MEG_runs)} run(s) remain.")

In [ ]:
MEG_runs

Define coregistration function(s):

In [ ]:
# --------------------------------------------------------------------
### COREGISTRATION FUNCTION (per subject_ID)
# --------------------------------------------------------------------

def run_coregistration_for_subject(subject_ID, subject_meg_runs, BEM_issues):

    try:
        print(f"\n=== Processing subject: {subject_ID} ===")

        # --------------------------------------------------------------
        # FreeSurfer directory (must exist from recon-all)
        # --------------------------------------------------------------
        fs_subject_dir = SUBJECTS_DIR / subject_ID
        if not fs_subject_dir.exists():
            message = f"FreeSurfer directory missing: {fs_subject_dir}"
            print("ERROR:", message)
            if HARD_STOP:
                raise FileNotFoundError(message)
            BEM_issues.append(subject_ID)
            return

        print(f"  - Found FreeSurfer directory: {fs_subject_dir}")

        # --------------------------------------------------------------
        # Filter MEG runs: keep those with resolved .fif paths
        # --------------------------------------------------------------
        subject_meg_runs = subject_meg_runs[subject_meg_runs["MEG_fullpath"].notna()]
        if subject_meg_runs.empty:
            print(f"  - No valid MEG files found for {subject_ID}. Skipping.")
            BEM_issues.append(subject_ID)
            return

        print(f"  - MEG runs to coregister: {len(subject_meg_runs)}")
        for _, row in subject_meg_runs.iterrows():
            print(f"      • {row['MEG_session_ID']} → {row['MEG_filename']}")

        # --------------------------------------------------------------
        # Output directory (one folder per subject)
        # --------------------------------------------------------------
        subject_output_dir = COREG_OUTPUT_DIR / subject_ID
        subject_output_dir.mkdir(parents=True, exist_ok=True)

        # ==============================================================
        # 1) Watershed + BEM solution (performed ONCE per subject)
        # ==============================================================
        print("\n  - Checking watershed/BEM surfaces...")

        bem_dir = fs_subject_dir / "bem"
        inner_skull = bem_dir / "inner_skull.surf"

        if OVERWRITE_WATERSHED or not inner_skull.exists():
            print("    → Running make_watershed_bem()")
            make_watershed_bem(
                subject=subject_ID,
                subjects_dir=str(SUBJECTS_DIR),
                volume="T1",
                preflood=BEM_FLOODING,
                overwrite=OVERWRITE_WATERSHED)
        else:
            print("    → Watershed surfaces already exist (skipping)")

        print("  - Creating BEM model + solution...")
        bem_model = mne.make_bem_model(
            subject=subject_ID,
            ico=BEM_ICO,
            conductivity=[BEM_CONNECTIVITY[2]],  # single-layer scalp conductivity
            subjects_dir=str(SUBJECTS_DIR))
        bem_solution = mne.make_bem_solution(bem_model)

        bem_solution_path = subject_output_dir / f"{subject_ID}_bem-sol.fif"
        mne.write_bem_solution(
            bem_solution_path,
            bem_solution,
            overwrite=OVERWRITE_WATERSHED)

        print(f"    → Saved BEM solution: {bem_solution_path}")

        # ==============================================================
        # 2) Loop over each MEG run (SESSION-SPECIFIC transforms)
        # ==============================================================
        for df_idx, row in subject_meg_runs.iterrows():

            session_ID = row["MEG_session_ID"]
            meg_label  = row["MEG_filename"]
            meg_path   = Path(row["MEG_fullpath"])

            print(f"\n  Processing MEG session: {session_ID}")
            print(f"    → MEG file: {meg_path}")

            if meg_path.suffix != ".fif":
                print("    → Skipping unsupported file format.")
                continue

            raw = mne.io.read_raw_fif(str(meg_path), preload=False)

            if "dig" not in raw.info:
                raise RuntimeError("Digitization data missing from MEG file")
            if "dev_head_t" not in raw.info:
                raise RuntimeError("Head transform missing from MEG file")

            # ---------------------------
            # Perform coregistration
            # ---------------------------
            coreg = mne.coreg.Coregistration(
                info=raw.info,
                subject=subject_ID,
                subjects_dir=str(SUBJECTS_DIR))
            coreg.fit_fiducials(verbose=True)
            coreg.fit_icp(n_iterations=COREG_ITER_NUM)
            trans = coreg.trans

            # ---------------------------
            # SESSION-SPECIFIC TRANS FILE
            # ---------------------------
            trans_filename = f"{subject_ID}_{session_ID}_trans.fif"
            trans_out = subject_output_dir / trans_filename

            mne.write_trans(
                trans_out,
                trans,
                overwrite=OVERWRITE_COREG)

            # Store for downstream pipeline stages
            MEG_runs.at[df_idx, "trans_fullpath"] = str(trans_out)

            print(f"    → Saved transform: {trans_out}")

    except Exception as err:
        print(f"\n!!! ERROR in coreg for subject {subject_ID} !!!")
        print(err)
        BEM_issues.append(subject_ID)
        if HARD_STOP:
            raise

In [ ]:
# --------------------------------------------------------------------
### DRIVER LOOP — RUN COREGISTRATION FOR ALL SUBJECTS
# --------------------------------------------------------------------

BEM_issues = []

# Iterate through each subject_ID that appears in MEG_runs
for subject_ID, subject_meg_runs in MEG_runs.groupby("subject_ID"):
    run_coregistration_for_subject(subject_ID, subject_meg_runs, BEM_issues)

# --------------------------------------------------------------------
### SAVE BEM ISSUES LIST
# --------------------------------------------------------------------
issues_file_path = COREG_OUTPUT_DIR / "BEM_issues.txt"

with open(issues_file_path, "w") as file:
    if len(BEM_issues) == 0:
        file.write("No BEM/coregistration issues detected.\n")
    else:
        for s in sorted(set(BEM_issues)):
            file.write(f"{s}\n")

print(f"\n[info] BEM issues list saved to: {issues_file_path}")

# --------------------------------------------------------------------
### PRINT SUMMARY
# --------------------------------------------------------------------
print("\n==================== SUMMARY ====================")
print(f"  Total subjects processed: {MEG_runs['subject_ID'].nunique()}")
print(f"  Subjects with BEM/coreg issues: {len(BEM_issues)}")

if BEM_issues:
    print("  → Problem subjects:")
    for s in sorted(set(BEM_issues)):
        print(f"       - {s}")
else:
    print("  → No errors.")
print("==================================================\n")
